# Week 08 - FastAPI Practice

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |
| 시트 기준 열 | FAST API 사이트 실습코드(4-29) |

이 노트북은 과제 요구사항을 학습용으로 재구성한 설명형 산출물이다. 원본 코드를 그대로 복사하지 않고, 같은 개념을 작은 로컬 예제로 다시 구현한다.

## 목표

FastAPI의 first steps, path parameter, query parameter, request body 개념을 실행 가능한 함수로 정리한다.

모든 코드는 외부 서비스 접속 없이 실행되도록 구성했다. 실제 API나 웹사이트를 사용할 때는 같은 처리 흐름에서 데이터 입력 부분만 교체하면 된다.

## 1. FastAPI 기본 구조

FastAPI 프로젝트는 `app = FastAPI()`를 만들고, 데코레이터로 URL과 함수를 연결한다. 현재 실행 환경에 FastAPI가 없어도 노트북 검증이 가능하도록 최소 대체 라우터를 둔다.

In [1]:
import pandas as pd

try:
    from fastapi import FastAPI
    FASTAPI_AVAILABLE = True
except Exception:
    FASTAPI_AVAILABLE = False

    class FastAPI:
        def __init__(self):
            self.routes = {}
        def get(self, path):
            def decorator(func):
                self.routes[("GET", path)] = func
                return func
            return decorator
        def post(self, path):
            def decorator(func):
                self.routes[("POST", path)] = func
                return func
            return decorator

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "Hello, FastAPI practice", "student": "이성민"}

result = await root()
assert result["student"] == "이성민"
print("FastAPI installed:", FASTAPI_AVAILABLE)
result

FastAPI installed: False


{'message': 'Hello, FastAPI practice', 'student': '이성민'}

## 2. Path Parameter

경로 매개변수는 URL의 일부가 함수 인자로 들어오는 방식이다. `item_id: int`처럼 타입을 적으면 FastAPI가 요청 값을 정수로 검증한다.

In [2]:
@app.get("/items/{item_id}")
async def read_item(item_id: int):
    return {"item_id": item_id, "double": item_id * 2}

path_result = await read_item(7)
assert path_result == {"item_id": 7, "double": 14}
path_result

{'item_id': 7, 'double': 14}

## 3. Query Parameter

URL의 `?skip=0&limit=10` 같은 부분은 쿼리 매개변수다. 목록 조회에서 페이지네이션과 검색 조건을 줄 때 많이 사용한다.

In [3]:
fake_db = [
    {"name": "Rocket", "category": "etl"},
    {"name": "Titanic", "category": "eda"},
    {"name": "Webtoon", "category": "scraping"},
    {"name": "FastAPI", "category": "backend"},
]

@app.get("/topics/")
async def read_topics(skip: int = 0, limit: int = 2, q: str | None = None):
    rows = fake_db
    if q:
        rows = [row for row in rows if q.lower() in row["name"].lower()]
    return rows[skip : skip + limit]

query_result = await read_topics(q="api", limit=5)
assert query_result[0]["name"] == "FastAPI"
query_result

[{'name': 'FastAPI', 'category': 'backend'}]

## 4. Request Body

POST나 PUT은 본문에 JSON 데이터를 담아 보낸다. 실제 FastAPI에서는 Pydantic 모델을 사용하지만, 여기서는 실행 안정성을 위해 표준 라이브러리 `dataclass`로 같은 구조를 설명한다.

In [4]:
from dataclasses import dataclass, asdict

@dataclass
class TopicInput:
    name: str
    category: str
    difficulty: int = 1

    def validate(self):
        if not self.name.strip():
            raise ValueError("name is required")
        if not 1 <= self.difficulty <= 5:
            raise ValueError("difficulty must be between 1 and 5")
        return self

@app.post("/topics/")
async def create_topic(topic: TopicInput):
    topic.validate()
    payload = asdict(topic)
    payload["message"] = "created"
    return payload

body_result = await create_topic(TopicInput(name="Request Body", category="backend", difficulty=3))
assert body_result["message"] == "created"
body_result

{'name': 'Request Body',
 'category': 'backend',
 'difficulty': 3,
 'message': 'created'}

## 5. 라우트 설계 요약

학습한 개념을 정리하면 GET은 조회, POST는 생성, path parameter는 특정 자원 지정, query parameter는 필터 조건, request body는 복합 데이터를 전달하는 방식이다.

In [5]:
route_summary = pd.DataFrame([
    {"concept": "first steps", "example": "GET /", "purpose": "API 상태 확인"},
    {"concept": "path parameter", "example": "GET /items/7", "purpose": "특정 자원 조회"},
    {"concept": "query parameter", "example": "GET /topics/?q=api", "purpose": "조건 필터링"},
    {"concept": "request body", "example": "POST /topics/", "purpose": "JSON 데이터 생성"},
])

assert len(route_summary) == 4
route_summary

,concept,example,purpose
0,first steps,GET /,API 상태 확인
1,path parameter,GET /items/7,특정 자원 조회
2,query parameter,GET /topics/?q=api,조건 필터링
3,request body,POST /topics/,JSON 데이터 생성
